In [15]:
import pandas as pd

In [16]:
df = pd.read_csv("../data/student_mental_health.csv")

df.head(3)

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium


In [17]:
df = df.drop(columns='Student_ID')
df.columns

Index(['Age', 'Gender', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week',
       'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Physical_Activity',
       'Financial_Stress', 'Family_Support', 'Attendance_Rate',
       'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History',
       'Mental_Health_Risk'],
      dtype='str')

In [18]:
y = df['Mental_Health_Risk']
X = df.drop(columns='Mental_Health_Risk')

display(y.shape)
display(X.columns)

(2000,)

Index(['Age', 'Gender', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week',
       'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Physical_Activity',
       'Financial_Stress', 'Family_Support', 'Attendance_Rate',
       'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History'],
      dtype='str')

In [19]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape : ', X_train.shape)
print('X_test shape : ', X_test.shape)
print('y_train shape : ', y_train.shape)
print('y_test shape : ', y_test.shape)

X_train shape :  (1600, 15)
X_test shape :  (400, 15)
y_train shape :  (1600,)
y_test shape :  (400,)


In [21]:
print("train : \n" , y_train.value_counts())
print("test : \n" , y_test.value_counts())



train : 
 Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64
test : 
 Mental_Health_Risk
High      136
Medium    132
Low       132
Name: count, dtype: int64


In [22]:
df.dtypes

Age                                 int64
Gender                                str
Year_of_Study                       int64
Sleep_Hours                       float64
Study_Hours_Per_Week              float64
GPA                               float64
Social_Media_Hours                float64
Screen_Time_Hours                 float64
Physical_Activity                     str
Financial_Stress                  float64
Family_Support                    float64
Attendance_Rate                   float64
Part_Time_Job                         str
Diet_Quality                          str
Previous_Mental_Health_History        str
Mental_Health_Risk                    str
dtype: object

In [27]:
numeric_columns = ["Age","Year_of_Study","Sleep_Hours","Study_Hours_Per_Week","GPA","Social_Media_Hours","Screen_Time_Hours"
                   ,"Financial_Stress","Family_Support","Attendance_Rate"]
ordinal_columns = ["Physical_Activity","Diet_Quality"]
binary_columns = ["Part_Time_Job","Previous_Mental_Health_History"]
nominal_columns = ["Gender"]


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler

from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

ordinal_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(
                                    categories=[["Low", "Moderate", "High"],
                                                ["Poor", "Average", "Good"]],
                                    handle_unknown="use_encoded_value",
                                    unknown_value=-1
                                    ))
])

binary_pipeline = Pipeline(
    steps=[
        ('encoder',OneHotEncoder(drop="if_binary", handle_unknown="ignore",sparse_output=False))
    ]
)

nominal_pipeline = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [29]:
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_columns),
    ("ordinal", ordinal_pipeline, ordinal_columns),
    ("binary", binary_pipeline, binary_columns),
    ("nominal", nominal_pipeline, nominal_columns)
])

In [30]:
preprocessor.fit(X_train)

X_train_ready = preprocessor.transform(X_train)
X_test_ready = preprocessor.transform(X_test)

In [32]:
import numpy as np
print("Train shape:", X_train_ready.shape)
print("Test shape:", X_test_ready.shape)
print("NaN left in train:", np.isnan(X_train_ready).sum())
print("NaN left in test:", np.isnan(X_test_ready).sum())

Train shape: (1600, 17)
Test shape: (400, 17)
NaN left in train: 0
NaN left in test: 0


In [ ]:
print(preprocessor.get_feature_names_out())

['numeric__Age' 'numeric__Year_of_Study' 'numeric__Sleep_Hours'
 'numeric__Study_Hours_Per_Week' 'numeric__GPA'
 'numeric__Social_Media_Hours' 'numeric__Screen_Time_Hours'
 'numeric__Financial_Stress' 'numeric__Family_Support'
 'numeric__Attendance_Rate' 'ordinal__Physical_Activity'
 'ordinal__Diet_Quality' 'binary__Part_Time_Job_Yes'
 'binary__Previous_Mental_Health_History_Yes' 'nominal__Gender_Female'
 'nominal__Gender_Male' 'nominal__Gender_Non-binary']


In [34]:
import pandas as pd

check = pd.DataFrame(X_train_ready, columns=preprocessor.get_feature_names_out())
print(check[["ordinal__Physical_Activity", "ordinal__Diet_Quality"]].describe())

       ordinal__Physical_Activity  ordinal__Diet_Quality
count                 1600.000000            1600.000000
mean                     0.862500               0.941250
std                      0.719491               0.730159
min                      0.000000               0.000000
25%                      0.000000               0.000000
50%                      1.000000               1.000000
75%                      1.000000               1.000000
max                      2.000000               2.000000


In [35]:
print(check[["numeric__Sleep_Hours", "numeric__GPA"]].describe().round(2))

       numeric__Sleep_Hours  numeric__GPA
count               1600.00       1600.00
mean                   0.00         -0.00
std                    1.00          1.00
min                   -2.85         -3.11
25%                   -0.68         -0.68
50%                   -0.04         -0.01
75%                    0.68          0.66
max                    2.95          3.20


In [36]:
print(check[["numeric__Sleep_Hours", "numeric__GPA"]].describe().round(2))

       numeric__Sleep_Hours  numeric__GPA
count               1600.00       1600.00
mean                   0.00         -0.00
std                    1.00          1.00
min                   -2.85         -3.11
25%                   -0.68         -0.68
50%                   -0.04         -0.01
75%                    0.68          0.66
max                    2.95          3.20
